## Compute the NIRCam AB magnitude at 5 $\sigma$ for a target integration time 

### Some NIRCam Docs
- [Readout Patterns](https://jwst-docs.stsci.edu/jwst-near-infrared-camera/nircam-instrumentation/nircam-detector-overview/nircam-detector-readout-patterns#gsc.tab=0)
- [Recommended Strategies](https://jwst-docs.stsci.edu/jwst-near-infrared-camera/nircam-observing-strategies/nircam-imaging-recommended-strategies#gsc.tab=0)

In [1]:
# Pandeia reference data
import os
os.environ["pandeia_refdata"] = "./ref_data/pandeia_data-2025.7-jwst"
os.environ["PYSYN_CDBS"] = "./ref_data/hlsp_reference-atlases_hst_multi_everything_multi_v16_sed/grp/redcat/trds"

# Exposure time calculator
from pandeia.engine.calc_utils import build_default_calc
from pandeia.engine.perform_calculation import perform_calculation

def tune_total_time(calc, target_exp, min_groups, max_groups, min_nint, max_nint):
    """ Adjust calc to get total_exposure_time ~= target_exp """
    current_groups = min_groups
    current_nint = min_nint

    # Iterate until we hit the target exposure time or exceed limits
    while current_nint <= max_nint and current_groups <= max_groups:

        # Set new detector settings
        calc['configuration']['detector']['ngroup'] = current_groups
        calc['configuration']['detector']['nint'] = current_nint

        # Perform new calculation
        rep = perform_calculation(calc)
        tot_exp = float(rep['information']['exposure_specification']['total_exposure_time'])  # seconds
        single_exp = float(rep['information']['exposure_specification']['exposure_time'])  # seconds
        single_int = single_exp / current_nint
        
        # Show progress
        print(f"[timing] nint={current_nint}, ngroup={current_groups}, tot_exp={tot_exp:.2f}s, single_exp={single_exp:.2f}s, single_int={single_int:.2f}s")

        # Check if close enough
        if tot_exp >= target_exp:
            break

        # Single integration limit exceeded (seconds)
        if single_int > max_single_int:
            current_groups = min_groups
            current_nint += 1
            continue        

        # Increase ngroups first, then nint
        if current_groups < max_groups:
            current_groups += 1
        else:
            current_groups = min_groups # reset
            current_nint += 1

    # One last report
    rep = perform_calculation(calc)
    return calc, rep

def solve_ab_depth(calc, target_snr=5.0, low=20.0, high=35.0, max_iter=30, tol=0.02):
    """
    Find AB magnitude where S/N ~= target_snr via bisection+secant hybrid.
    Returns (depth_ab, final_snr_at_depth)
    """

    # Bisection method
    for _ in range(max_iter):
        check_mag = 0.5 * (low + high)

        # At this mag, what's the S/N?
        calc['scene'][0]['spectrum']['normalization']['norm_flux'] = check_mag
        snr = perform_calculation(calc)['scalar']['sn']

        # Adjust bounds
        if snr >= target_snr:
            low = check_mag # Too bright
        else:
            high = check_mag # Too faint

        # Return if S/N close
        if abs(snr - target_snr) < tol:
            return check_mag, snr

# Telescope
telescope = 'jwst'
instrument = 'nircam'
mode = 'lw_imaging'
filter = 'f356w'
readout_pattern = "medium8" # "deep2", "deep8", "medium8", "shallow4", "bright2", "rapid"

# Target parameters
target_exp = 3600*6 # total exposure time in seconds
target_snr = 5.0 # 5-sigma depth

# Tuning exposure parameters
min_groups = 5 # recommended, 1 for rapid
max_groups = 10 # 10 for all, except deep2 and deep8 = 20 max groups, rapid = max 2
min_nint = 1
max_nint = 10 # max allowed
nexp = 4
max_single_int = 1000 # seconds, avoid too-long single integrations. Longer for deep2,8. 1000s for all else

# Flux parameters
flux_unit = 'abmag' # flux unit
magnitude = 26.0 # units above

# 1) Set default calculation
calc = build_default_calc(telescope, instrument, mode)

# 2) Set instrument/filter and point-source normalization in AB for the same bandpass
calc['configuration']['instrument']['filter'] = filter
calc['scene'][0]['shape']['geometry'] = 'point'
calc['scene'][0]['spectrum']['normalization'] = {
    'type': f'{telescope}',
    'bandpass': f'{instrument},{mode},{filter}',
    'norm_flux': magnitude,  # neutral starting guess (AB mag)
    'norm_fluxunit': flux_unit
}

# 3) Detector settings (will tune nint below to hit time target)
calc['configuration']['detector']['readout_pattern'] = readout_pattern  # pandeia uses lowercase names
calc['configuration']['detector']['ngroup'] = min_groups
calc['configuration']['detector']['nint'] = min_nint
calc['configuration']['detector']['nexp'] = nexp

# 4) Background level
calc['background_level'] = 'low'  # 'low', 'medium', 'high'

# 5) Tune total exposure time to ~ target_exp by scaling nint
calc, rep_timing = tune_total_time(calc, target_exp, min_groups, max_groups, min_nint, max_nint)
exp = rep_timing['information']['exposure_specification']
print(f"[timing] Achieved: total_exposure_time={exp['total_exposure_time']:.2f} s "
        f"(nint={exp['nint']}, ngroup={exp['ngroup']}, readout={exp['readout_pattern']})")

# 6) Solve for AB depth at target S/N
depth_ab, sn_at_depth = solve_ab_depth(calc, target_snr)

print(f"\n=== {telescope}/{instrument}: {target_snr}sig depth @ {target_exp} s ===")
print(f"Depth: {depth_ab:.3f} AB")
print(f"S/N at depth: {sn_at_depth:.2f}")

tot_exp_time = rep_timing['information']['exposure_specification']['total_exposure_time']
n_groups = rep_timing['information']['exposure_specification']['ngroup']
n_int = rep_timing['information']['exposure_specification']['nint']
n_exp = rep_timing['information']['exposure_specification']['nexp']
readout = rep_timing['information']['exposure_specification']['readout_pattern']
print(f"Total exposure time: {tot_exp_time:.2f} seconds (ngroup={n_groups}, nint={n_int}, nexp={n_exp}, readout={readout})")

[timing] nint=1, ngroup=5, tot_exp=2061.46s, single_exp=515.36s, single_int=515.36s
[timing] nint=1, ngroup=6, tot_exp=2490.93s, single_exp=622.73s, single_int=622.73s
[timing] nint=1, ngroup=7, tot_exp=2920.40s, single_exp=730.10s, single_int=730.10s
[timing] nint=1, ngroup=8, tot_exp=3349.87s, single_exp=837.47s, single_int=837.47s
[timing] nint=1, ngroup=9, tot_exp=3779.34s, single_exp=944.84s, single_int=944.84s
[timing] nint=1, ngroup=10, tot_exp=4208.81s, single_exp=1052.20s, single_int=1052.20s
[timing] nint=2, ngroup=5, tot_exp=4165.87s, single_exp=1041.47s, single_int=520.73s
[timing] nint=2, ngroup=6, tot_exp=5024.81s, single_exp=1256.20s, single_int=628.10s
[timing] nint=2, ngroup=7, tot_exp=5883.75s, single_exp=1470.94s, single_int=735.47s
[timing] nint=2, ngroup=8, tot_exp=6742.69s, single_exp=1685.67s, single_int=842.84s
[timing] nint=2, ngroup=9, tot_exp=7601.63s, single_exp=1900.41s, single_int=950.20s
[timing] nint=2, ngroup=10, tot_exp=8460.57s, single_exp=2115.14s, s